# Memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
from typing import Dict, Literal, Optional
from pydantic import BaseModel, Field, RootModel


class CapitalInfo(BaseModel):
    country: Optional[str] = Field(default=None)
    capital: Optional[str] = Field(default=None)
    source: Optional[Literal["tool", "AI fallback", "tool-not-found"]] = Field(default=None)
    explanation: Optional[str] = Field(default=None)


class Response(BaseModel):
    result: Dict[str, CapitalInfo] = Field(
        description="Dictionary where key is question number as string and value is the answer object"
    )

# MiddleWare
- The middleware ecosystem provides:
    - Execution Environment: tools, filesystem, sandboxes, code Execution
    - Context management: Summarization, memory, skills and prompt caching
    - Planning and delegation: Todo lists and subagents for parallel, isolated work
    - Fault tolerance: Retries, fallbacks can call limits
    - Guardrails: PII detectiona and content control
    - Steering: HITL

## Execution environment
provide the agent a workspace to make decision
- gives the agent a workspace
    - tools it can call
    - filesystem for reading and writing accross turns
    - code execution for running scripts or shell commands

In [ ]:
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware

subagents_list=[
    {
        "name":"model1",
        "system_prompt":"",
    },
    {
        "name":"model2",
        "system_prompt":"",
    }
]

middleware_list=[
    ModelRetryMiddleware(max_retries=3),
    ToolRetryMiddleware(max_retries=2),
    SubAgentMiddleware(
        subagents=subagents_list
    )
]

# Message

- Message Represent input and output of model
- Text Prompt: straight forward Raw string
    - single, standalone request
    - dont persist conversation history
- Message Prompt
    - Usages
        - Multi-turn , multi model conversations
    - Message Contains
        - Role: Identify the message type (system or user)
        - Content: Represent the actual content of Message (text, image, audio)
        - Metadata: Optional metadata
    - Message Type
        - System Message: Define model's behaviour
        - Human Message: User interaction or input to model
        - AI Message: Responses generated by the model
        - Tool Message: Reprsent the output of tool calls
    

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]
response = model.invoke(messages)


# Context
- use to pass per-run configuration

In [2]:
from dataclasses import dataclass
@dataclass
class Context:
    user_id:str

# Agent
- agent=model+harness (skills, tools, contest, subagent, system prompt, memory)
- it is model calling tools in a loop until a given task is complete

In [ ]:
agent = create_agent(
    name="CapitalFinder",
    model="gemini-3.1-flash-lite", #you can also pass standalone model object
    tools=[get_capital],
    system_prompt="you are a helpful assistant that provides capital information for countries. If the country is not found, you can provide fallback information if allowed.",
    response_format=Response,
    middleware=middleware_list,
    checkpointer=InMemorySaver(),
    context_schema=Context
)

# Invocation to Agent

- it can be done in 3 ways
    - invoke
    - stream
    - batch

## invoke

A follow-up turn on the same conversation: reuse the same thread_id to keep history
- thread_id scopes the conversation (message history, checkpoints), while context carries per-run data your tools and middleware read at invocation time.

In [ ]:
from langchain_core.utils.uuid import uuid7

content = """
Answer each question with both the capital and the source.

1) What is the capital of India?
2) What is the capital of Brazil?
3) What is the capital of France?
4) What should you do if asked about India?
"""

config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the capital of japan?"}]},
    config=config1,
    context=Context(user_id="user123")
)
print(result['messages'][-1].content[0]['text'])

# A follow-up turn on the same conversation: reuse the same thread_id to keep history
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what about india?"}]},
    config=config1,
    context=Context(user_id="user123")
)
print(result['messages'][-1].content[0]['text'])


Different conversation: different thread_id history not saved

In [ ]:
from langchain_core.utils.uuid import uuid7
config2 = {"configurable": {"thread_id": str(uuid7())}}
# Different conversation: different thread_id history not saved
result = agent.invoke(
    {"messages": [{"role": "user", "content": "find for france?"}]},
    config=config2,
)
print(result3['messages'][-1].content[0]['text'])


# Streaming
- provide intermidiate progress update before completion

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

content = """
1) What is the capital of India?
2) What is the capital of Brazil?
3) What is the capital of France?
4) What should you do if asked about India?
"""

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": content}]},
    version="v3",
    config={
        "recursion_limit": 8,
        "configurable": {
            "thread_id": "capital-stream-debug-1"
        }
    },
)

for snapshot in stream.values:
    latest_message = snapshot["messages"][-1]

    if isinstance(latest_message, HumanMessage):
        print(f"\nUSER:\n{latest_message.content}")

    elif isinstance(latest_message, AIMessage):
        # Case 1: Agent wants to call a tool
        if latest_message.tool_calls:
            print("\nAGENT IS CALLING TOOL:")
            for tc in latest_message.tool_calls:
                print(f"Tool name: {tc['name']}")
                print(f"Tool args: {tc['args']}")

        # Case 2: Agent gives final answer
        elif latest_message.content:
            print("\nFINAL AGENT ANSWER:")
            print(latest_message.content)

    elif isinstance(latest_message, ToolMessage):
        print("\nTOOL RESULT:")
        print(f"Tool name: {latest_message.name}")
        print(f"Tool output: {latest_message.content}")

## Batch

In [ ]:
## Not Tested Yet

questions = [
    "What is the capital of India?",
    "What is the capital of Brazil?",
    "What is the capital of France?",
    "What should you do if asked about India?",
]

inputs = [{"messages": [{"role": "user", "content": q}]} for q in questions]

responses = agent.batch(inputs, config={"recursion_limit": 20})

for i, res in enumerate(responses, 1):
    print(f"\n--- Response {i} ---")
    print(res["messages"][-1].content)